In [1]:
import ee
import luma_ge
import geemap
ee.Authenticate() #force=True use for re-authentication
ee.Initialize()

c:\Users\AFahrezi\AppData\Local\anaconda3\envs\luma-lite\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
#Area of Interest (AOI) Ogan Ilir, South Sumatra, Indonesia
aoi = ee.FeatureCollection("projects/ee-agilfahrezy60/assets/AOI_Oganilir").geometry()
#Reference Data
ref_data = ee.FeatureCollection("projects/ee-agilfahrezy60/assets/Ref_data_ogan_ilir_2016")
all_samples = ref_data.map(lambda f: f.set({
    'class_id': f.get('id_epistem'),
    'class_name': f.get('name_epistem')
}))

In [3]:
# Reference year, closest match to the reference data collection (2016-2017)
YEAR = 2015
#GLAD Global Land Cover DATASETS — primary element source
#Ocean mask (apply to all GLAD layers)
landmask = ee.Image("projects/glad/OceanMask").lte(1)
# Combined LCLUC composite (used as fallback for classes GLAD has no dedicated layer for, e.g. shrub, herbaceous, bare soil)
glad_lc = ee.Image('projects/glad/GLCLU2020/v2/LCLUC_2015') \
    .updateMask(landmask)
#cropland dataset
glad_cropland = ee.ImageCollection('users/potapovpeter/Global_cropland_2015').mosaic() \
    .updateMask(landmask).unmask(0)
#build up dataset
glad_builtup = ee.Image('projects/glad/GLCLU2020/Builtup_type') \
    .updateMask(landmask).unmask(0)
#glad surface water
# Use the GLAD annual surface water single-band dataset, not the 3-band RGB product.
#glad_sw = ee.Image( 'projects/glad/water/C2/dynamic_rgb_99_25') \
#   .updateMask(landmask).unmask(0)

## Tree Element and Properties

In [4]:
# --- sort_ID 1: tree / elementPresenceType  (binary 0/1) ---
# GLAD dedicated forest extent layer: pixels with forest height >= 5 m
# Note: Forest_extent_2020 pixel value = 1 where forest present; already binary
tree_ht = ee.Image('projects/glad/GLCLU2020/Forest_height_2020')
tree_presence = (
    tree_ht.gt(10)   
    .rename('treePresenceType')
)
# --- sort_ID 2: tree / cover  (continuous 0-100%) ---
#GFCC30TC (NASA MEASURES)
tree_cover = (ee.ImageCollection('NASA/MEASURES/GFCC/TC/v3')
              .filter(ee.Filter.date(f'{YEAR}-01-01', f'{YEAR}-12-31'))
              .select('tree_canopy_cover')
              .mean()
              .rename('treecover'))
# --- sort_ID 3: woodyGrowthForm / height  (continuous, metres) ---
tree_height = (ee.Image('projects/glad/GLCLU2020/Forest_height_2020')
               .updateMask(landmask)
               .unmask(0)
               .rename('treeheight'))
# --- sort_ID 4: leafCharacterSizeType
lai_mean = ee.ImageCollection('MODIS/061/MOD15A2H') \
    .filterDate(f'{YEAR}-01-01', f'{YEAR}-12-31') \
    .select('Lai_500m') \
    .mean() \
    .multiply(0.1) \
    .rename('lai_mean')
# --- sort_ID 5-7: tree horizontal spreading, temporal type, length ---
# There's no availible proxy for these elements


In [5]:
# --- sort_ID 9: woodyGrowthForm / woodyLeafPhenology (Evergreen/Deciduous) ---
# Use MODIS MCD12Q1 LC_Type5:
#   1 = evergreen needleleaf, 2 = evergreen broadleaf,
#   3 = deciduous needleleaf, 4 = deciduous broadleaf
woody_leaf_phenology = (ee.ImageCollection('MODIS/061/MCD12Q1')
                        .filterDate(f'{YEAR}-01-01', f'{YEAR+1}-01-01')
                        .first()
                        .select('LC_Type5')
                        .rename('treewoodyLeafPhenology'))

# --- sort_ID 10: woodyGrowthForm / woodyLeafType (Broadleaf/Needleleaf) ---
# Same MODIS layer; values 1,3 = needleleaf; 2,4 = broadleaf
# Remap: 1,3 → 1 (needleleaf), 2,4 → 2 (broadleaf), others → 0
woody_leaf_type = (woody_leaf_phenology
                   .remap([1, 2, 3, 4], [1, 2, 1, 2], defaultValue=0)
                   .rename('woodyLeafType'))


## Build Up Presence

In [6]:
# BUILT-UP
# Template rows: sort_ID 20, 21, 22, 23, 24, 25
# --- sort_ID 20, 22, 23: builtup / nonLinear / building presence (binary) ---
# GLAD dedicated built-up layer:
#   Value 1 = stable built-up area 2000-2020
#   Value 2 = built-up expansion 2000-2020
# since there's no distinction between non-settlement built-up (e.g. plaza, parking lot) vs settlement built-up (building footprint) 
builtup_presence = glad_builtup.gte(1).unmask(0).rename('builtUpPresenceType')
# --- sort_ID 21: constructionMaterial ---
# NOT available . Omitted.
# --- sort_ID 24-25: linearSurface presence and type ---
# NOT available Roads/railways require OSM or similar. Omitted.


## Non Linear Surface Element

In [7]:
#Global Building Surface Layer (GHSL) provides complementary built-up surface data, including non-residential built-up areas
ghsl = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S_10m/2018')
built_surface_m2 = ghsl.select('built_surface').rename('built_surface_m2')
#build up surface non residential
built_nres_m2    = ghsl.select('built_surface_nres').rename('built_nres_m2')

## Shrub Element

In [8]:
# Template rows: sort_ID 8, 11, 12
# --- sort_ID 8: shrub / elementPresenceType (binary 0/1) ---
# Best proxy: LCLUC composite value 20 = short woody vegetation
#Broad approximation:
#shrub_presence = glad_lc.eq(20).rename('shrubPresenceType')
#alternative proxy using tree height and absence of other land cover types (cropland, built-up)
#woody growth form = shrub if height > 0 and < 5 m, and not cropland or built-up
shrub_presence = (
    tree_height.gt(0)
    .And(tree_height.lt(5))
    .And(glad_cropland.eq(0))
    .And(builtup_presence.eq(0))
    .rename('shrubPresenceType')
)


## Water element

In [9]:
# Values: 1 = permanent water, 2 = seasonal water, 3 = ephemeral water
# --- sort_ID 29: waterBody / elementPresenceType (binary) ---
#Any water presence type counts as water presence for this binary variable
#water_presence = glad_sw.gte(1).rename('waterBodyPresence')
# --- sort_ID 30: waterBody / dynamics (flowing/standing) ---
#No data available
# JRC GSW transition layer provides the best available proxy.
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
# Use JRC occurrence as continuous water metric
water_pres  = gsw.select('occurrence').rename('water_presence')
# Seasonality = number of months water present per year (0-12)
water_seasonality = gsw.select('seasonality').rename('waterSeasonality')
    
# --- sort_ID 32: waterBody / periodVariationType (categorical) ---
#GLAD surface water layer provides a categorical variable for water presence type, based on intra-annual variability:
#   Value 1 = permanent water   → maps to "permanent"
#   Value 2 = seasonal water    → maps to "seasonal"
#   Value 3 = ephemeral water   → maps to "atmospheric/ephemeral"
#water_period_type = glad_sw.rename('waterPeriodVariationType')
# Integer meaning: 0=no water, 1=permanent, 2=seasonal, 3=ephemeral

## Herbs Element

In [10]:
#HERBACEOUS (BY EXCLUSION)
# --- sort_ID 11-12: shrub horizontal spreading, temporal type ---
# NOT available, Omitted.
# ELEMENT 3: HERBACEOUS / GRAMINOID / FORB
# Template rows: sort_ID 13, 14, 15, 16, 17, 18, 19
# --- sort_ID 13: herbaceousGrowthForm / elementPresenceType (binary) ---
# GLAD proxy: LCLUC value 20 (short veg) OR 30 (sparse veg/open herb)
# VEGETATION MASK
# select pixels contain non-woody vegetation
vegetation_mask = (
    water_pres.eq(0)
    .And(builtup_presence.eq(0))
)
herb_presence = (
    vegetation_mask
    .And(tree_presence.eq(0))
    .And(shrub_presence.eq(0))
    .And(glad_cropland.eq(0))
    .rename('herbaceousPresenceType')
)

## Graminoid

In [11]:
# GRAMINOID (NDVI SEASONALITY PROXY)
# Vegetation NDVI statistics (complement for herb/graminoid density)
#use landsat NDVI annual composites to capture intra-annual variability, which can be a proxy for herbaceous/graminoid presence and density
ndvi_col   = (ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_ANNUAL_NDVI')
              .filterDate(f'{YEAR}-01-01', f'{YEAR}-12-31')
              .select('NDVI'))
#NDVI stats
ndvi_mean  = ndvi_col.mean().rename('ndvi_mean')
ndvi_min   = ndvi_col.min().rename('ndvi_min')
ndvi_max   = ndvi_col.max().rename('ndvi_max')
ndvi_std   = ndvi_col.reduce(ee.Reducer.stdDev()).rename('ndvi_std')
ndvi_amp = ndvi_max.subtract(ndvi_min)
graminoid_presence = (
    ndvi_amp.gt(0.25)
    .And(tree_presence.eq(0))
    .And(shrub_presence.eq(0))
    .And(glad_cropland.eq(0))
    .And(builtup_presence.eq(0))
    .rename('graminoidPresenceType')
)

## Natural Surface Element

In [12]:
# --- sort_ID 26: naturalSurface / elementPresenceType (binary) ---
# Derived: any pixel that is NOT built-up, NOT water, NOT Cropland = natural surface
#Add NDVI
natural_surface_presence = (
    # NOT artificial
    builtup_presence.eq(0)
    .And(glad_cropland.eq(0))
    #NOT water
    .And(water_pres.eq(0))
    #ABIOTIC: low vegetation signal
    .And(ndvi_mean.lt(0.2))   
    .rename('naturalSurface_presence')
)

## Baresoil

In [13]:
# BARE SOIL
# Template rows: sort_ID 26, 27, 28
# --- sort_ID 27: bareSoil / elementPresenceType (binary) ---
#GLAD bare soil proxy layer, less than 5% vegetation cover and low NDVI
bare_soil_presence = (
    glad_lc.lt(5)
    .And(ndvi_mean.lt(0.2))
    .rename('bareSoilPresence')
)

## Vegetation Artificially

In [14]:
#ELEMENT 7: VEGETATION ARTIFICIALITY
# --- sort_ID 39: vegetationArtificiality (natural vs cultivated) ---
# GLAD dedicated cropland layer:
#   projects/glad/GLCLU2020/Cropland
#   Value 1 = cropland (cultivated/managed herbaceous)
#assumses cropland with human intervention
vegetation_artificiality = (
    glad_cropland.gte(1)
    .Or(builtup_presence.eq(1))
    .rename('vegetationArtificiality')
)
#Complementary data for crop presence and intensity, since GLAD cropland layer is binary and may miss some cropland areas
# Crop intensity from GCI30 (complements GLAD cropland)
gci30          = ee.ImageCollection("projects/sat-io/open-datasets/GCI30").median()
# Cropping intensity (1–3 crops/year)
crop_intensity = (
    gci30.select('b1')
    .where(gci30.select('b1').eq(-1), 0)
    .rename('crop_intensity')
)
# Number of crop cycles (cleaned)
crop_cycles = (
    gci30.select('b2')
    .where(gci30.select('b2').eq(127), 0)
    .rename('crop_cycles')
)

In [15]:
# ASSEMBLE FINAL STACK
# =============================================================================
# Only include layers that have actual data — omitted elements are documented
# above as NOT AVAILABLE in GLAD.

stack = ee.Image.cat([
    # ── TREE BLOCK ──────────────────────────────────────────────────
    tree_presence,          # sort_ID 1:  binary, GLAD Forest_extent
    tree_cover,             # sort_ID 2:  continuous %, GFCC30TC (GLAD has no cover)
    tree_height,            # sort_ID 3:  continuous m, GLAD Forest_height_2020
    lai_mean,               # sort_ID 4:  continuous, MODIS proxy for leaf size
    woody_leaf_phenology,   # sort_ID 9:  categorical, MODIS MCD12Q1 LC_Type5
    woody_leaf_type,        # sort_ID 10: categorical, derived from above
    # ── SHRUB BLOCK ─────────────────────────────────────────────────
    shrub_presence,         # sort_ID 8:  binary, GLAD LCLUC val=20 (proxy)
    # ── HERBACEOUS / GRAMINOID BLOCK ────────────────────────────────
    herb_presence,          # sort_ID 13: binary, GLAD LCLUC val=20|30 (proxy)
    graminoid_presence,     # sort_ID 15: binary, GLAD LCLUC val=30 (proxy)
    ndvi_mean,              # vegetation density proxy
    ndvi_min,               # dry-season vegetation signal
    ndvi_max,               # peak-season vegetation signal
    ndvi_std,               # phenological variability
    # ── BUILT-UP BLOCK ──────────────────────────────────────────────
    builtup_presence,       # sort_ID 20: binary, GLAD Builtup_type
    built_nres_m2,          # sort_ID 22: same GLAD laye
    # ── NATURAL SURFACE / BARE SOIL BLOCK ───────────────────────────
    natural_surface_presence,    # sort_ID 26: derived from GLAD layers
    bare_soil_presence,          # sort_ID 27: GLAD LCLUC val=40|35 (proxy)
    # ── WATER BODY BLOCK ────────────────────────────────────────────
    water_pres,         # sort_ID 29: binary, GLAD SW
    #water_period_type,      # sort_ID 32: categorical, GLAD SW (0/1/2/3)
    #water_occurrence,       # continuous %, JRC GSW (complements GLAD)
    water_seasonality,      # months/year, JRC GSW (complements GLAD)
    # ── VEGETATION ARTIFICIALITY ─────────────────────────────────────
    vegetation_artificiality,    # sort_ID 39: GLAD Cropland proxy
    crop_intensity,         # GCI30 crop intensity
    crop_cycles,            # GCI30 crop cycles
])

print("Stack band names:")
print(stack.bandNames().getInfo())

# =============================================================================
# EXTRACT RS VALUES AT TRAINING POINT LOCATIONS
# =============================================================================

extracted = stack.reduceRegions(
    collection=all_samples,
    reducer=ee.Reducer.mean(),
    scale=30,
    tileScale=4,
)

Stack band names:
['treePresenceType', 'treecover', 'treeheight', 'lai_mean', 'treewoodyLeafPhenology', 'woodyLeafType', 'shrubPresenceType', 'herbaceousPresenceType', 'graminoidPresenceType', 'ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std', 'builtUpPresenceType', 'built_nres_m2', 'naturalSurface_presence', 'bareSoilPresence', 'water_presence', 'waterSeasonality', 'vegetationArtificiality', 'crop_intensity', 'crop_cycles']


In [16]:
selectors = [
'ID_epistem',
'name_epist',
'treePresenceType', 
 'treecover', 
 'treeheight', 
 'lai_mean', 
 'treewoodyLeafPhenology', 
 'woodyLeafType', 
 'shrubPresenceType', 
 'herbaceousPresenceType', 
 'graminoidPresenceType', 
 'ndvi_mean', 
 'ndvi_min', 
 'ndvi_max', 
 'ndvi_std', 
 'builtUpPresenceType', 
 'built_nres_m2', 
 'naturalSurface_presence', 
 'bareSoilPresence', 
 'water_presence', 
 'waterSeasonality', 
 'vegetationArtificiality', 
 'crop_intensity', 
 'crop_cycles']
task_csv = ee.batch.Export.table.toDrive(
    collection=extracted,
    description='LCCS_Features_CSV',
    folder='GEE_exports',
    fileNamePrefix='LCCS_elements_table',
    fileFormat='CSV',
    selectors=selectors
)

task_csv.start()
task_shp = ee.batch.Export.table.toDrive(
    collection=extracted,
    description='LCCS_Features_SHP',
    folder='GEE_exports',
    fileNamePrefix='LCCS_elements_shp',
    fileFormat='SHP'
)

task_shp.start()

In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=stack.toFloat(),
    description='Source_LCCS_Elements_Image_rev',
    folder='Earth Engine',
    fileNamePrefix='Source_LCCS_Elements_Image_rev',
    scale=30,
    region=aoi,  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
